# Portugal Data Intelligence — Exploratory Data Analysis
*A comprehensive exploration of Portuguese macroeconomic indicators (2010–2025)*

This notebook walks through the analytical process from raw data exploration to cross-pillar insights, demonstrating data engineering, statistical analysis, and economic reasoning.

**Data pillars covered:**
1. GDP (quarterly)
2. Unemployment (monthly)
3. Interest Rates (monthly)
4. Credit to the Economy (monthly)
5. Inflation (monthly)
6. Public Debt (quarterly)

**Key economic periods:**
| Period | Years | Context |
|--------|-------|---------|
| Pre-crisis | 2010–2011 | Post-GFC recovery |
| Troika | 2012–2014 | EU/IMF bailout programme |
| Recovery | 2015–2019 | Structural adjustment and growth |
| COVID | 2020 | Pandemic shock |
| Post-COVID | 2021–2025 | Recovery, inflation, normalisation |

In [ ]:
# ── Setup and Imports ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sqlite3
from scipy import stats
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# Project imports
sys.path.insert(0, "..")
from config.settings import DATABASE_PATH
from src.analysis.significance_tests import (
    test_trend_significance,
    test_period_comparison,
    test_structural_break,
    test_granger_proxy,
    PERIODS,
)

# ── Design System ────────────────────────────────────────────────────────────
from src.reporting.shared_styles import (
    CHART_COLORS as COLORS,
    CHART_PRIMARY, CHART_SECONDARY, CHART_ACCENT, CHART_POSITIVE,
    CHART_NEUTRAL, CHART_DARK_TEXT, CHART_LIGHT_TEXT, CHART_PURPLE,
    PERIOD_COLORS, ZONE_CAUTION,
    CHART_FONT_SIZES, CHART_DPI, CHART_DISPLAY_DPI,
    CHART_GRID_ALPHA, CHART_LEGEND_FRAMEALPHA,
    apply_chart_style,
)

# Apply project-wide chart styling (replaces inline rcParams)
apply_chart_style()
plt.rcParams.update({"figure.figsize": (12, 5)})

# Map PERIODS keys (snake_case) to PERIOD_COLORS keys (Title Case)
_PERIOD_KEY_MAP = {
    "pre_crisis": "Pre-crisis",
    "troika": "Troika",
    "recovery": "Recovery",
    "covid": "COVID",
    "post_covid": "Post-COVID",
}

def period_color(period_key):
    """Look up color for a period key (supports both snake_case and Title Case)."""
    mapped = _PERIOD_KEY_MAP.get(period_key, period_key)
    return PERIOD_COLORS.get(mapped, CHART_NEUTRAL)

print("Setup complete. Database path:", DATABASE_PATH)

In [ ]:
# ── Load all data from the SQLite database ──────────────────────────────────
conn = sqlite3.connect(str(DATABASE_PATH))

gdp = pd.read_sql(
    "SELECT g.*, d.year, d.quarter "
    "FROM fact_gdp g JOIN dim_date d ON g.date_key = d.date_key "
    "ORDER BY g.date_key", conn)

unemployment = pd.read_sql(
    "SELECT u.*, d.year, d.month "
    "FROM fact_unemployment u JOIN dim_date d ON u.date_key = d.date_key "
    "ORDER BY u.date_key", conn)

interest_rates = pd.read_sql(
    "SELECT ir.*, d.year, d.month "
    "FROM fact_interest_rates ir JOIN dim_date d ON ir.date_key = d.date_key "
    "ORDER BY ir.date_key", conn)

credit = pd.read_sql(
    "SELECT c.*, d.year, d.month "
    "FROM fact_credit c JOIN dim_date d ON c.date_key = d.date_key "
    "ORDER BY c.date_key", conn)

inflation = pd.read_sql(
    "SELECT inf.*, d.year, d.month "
    "FROM fact_inflation inf JOIN dim_date d ON inf.date_key = d.date_key "
    "ORDER BY inf.date_key", conn)

public_debt = pd.read_sql(
    "SELECT pd.*, d.year, d.quarter "
    "FROM fact_public_debt pd JOIN dim_date d ON pd.date_key = d.date_key "
    "ORDER BY pd.date_key", conn)

datasets = {
    "GDP": gdp, "Unemployment": unemployment, "Interest Rates": interest_rates,
    "Credit": credit, "Inflation": inflation, "Public Debt": public_debt,
}

print("Data loaded successfully.")
for name, df in datasets.items():
    print(f"  {name:18s}  {df.shape[0]:>5d} rows  x  {df.shape[1]:>2d} cols")

## 1. Data Overview

A quick look at each dataset: column types, sample rows, and summary statistics.

In [ ]:
# ── Summary statistics for each dataset ──────────────────────────────────────
for name, df in datasets.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumn types:\n{df.dtypes.to_string()}")
    print(f"\nFirst 3 rows:")
    display(df.head(3))
    print(f"\nDescriptive statistics:")
    display(df.describe().round(2))

## 2. GDP Deep Dive

GDP is the headline indicator of economic health. We examine level evolution, growth-rate distribution, quarterly seasonality, and whether the differences across economic periods are statistically significant.

In [ ]:
# ── Identify the main GDP value column dynamically ──────────────────────────
_skip = {"date_key", "year", "quarter", "source_key"}
gdp_num_cols = [c for c in gdp.select_dtypes(include=[np.number]).columns if c not in _skip]
gdp_val_col = next((c for c in ["gdp_nominal", "nominal_gdp", "gdp_real", "real_gdp", "gdp", "value"]
                     if c in gdp_num_cols), gdp_num_cols[0] if gdp_num_cols else None)

# Annual aggregation
gdp_annual = gdp.groupby("year")[gdp_val_col].mean().reset_index()
gdp_annual["growth_rate"] = gdp_annual[gdp_val_col].pct_change() * 100

# ── Chart 1: GDP evolution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) GDP level over time
ax = axes[0]
ax.plot(gdp_annual["year"], gdp_annual[gdp_val_col], marker="o", linewidth=2, color=CHART_PRIMARY)
ax.set_title("GDP Evolution (annual average)")
ax.set_xlabel("Year")
ax.set_ylabel(gdp_val_col.replace("_", " ").title())
ax.axvspan(2012, 2014, alpha=0.12, color=CHART_ACCENT, label="Troika")
ax.axvspan(2020, 2020.5, alpha=0.15, color=ZONE_CAUTION, label="COVID")
ax.legend(loc="upper left")

# (b) Growth rate distribution
ax = axes[1]
growth = gdp_annual["growth_rate"].dropna()
ax.hist(growth, bins=12, edgecolor="white", color=CHART_SECONDARY, alpha=0.85)
ax.axvline(growth.mean(), color=CHART_ACCENT, linestyle="--", label=f"Mean = {growth.mean():.2f}%")
ax.axvline(0, color=CHART_DARK_TEXT, linestyle="-", linewidth=0.8)
ax.set_title("Distribution of Annual GDP Growth Rates")
ax.set_xlabel("Growth rate (%)")
ax.set_ylabel("Frequency")
ax.legend()

# (c) Quarterly seasonality
ax = axes[2]
if "quarter" in gdp.columns:
    gdp_box = gdp[[gdp_val_col, "quarter"]].dropna()
    gdp_box.boxplot(column=gdp_val_col, by="quarter", ax=ax,
                     patch_artist=True,
                     boxprops=dict(facecolor=CHART_SECONDARY, alpha=0.6))
    ax.set_title("GDP by Quarter")
    ax.set_xlabel("Quarter")
    ax.set_ylabel(gdp_val_col.replace("_", " ").title())
    plt.suptitle("")  # remove auto title from boxplot

plt.tight_layout()
plt.savefig("../reports/ad_hoc/gdp_deep_dive.png", bbox_inches="tight", dpi=CHART_DPI)
plt.show()
print(f"Primary GDP column: {gdp_val_col}")

In [ ]:
# ── GDP period comparison with t-test results ───────────────────────────────
gdp_period_result = test_period_comparison(gdp_annual, gdp_val_col, "year")

print("GDP Period Comparison (ANOVA + pairwise Welch t-tests)")
print("=" * 60)
print(f"\nANOVA: F = {gdp_period_result['anova']['f_statistic']}, "
      f"p = {gdp_period_result['anova']['p_value']}, "
      f"significant = {gdp_period_result['anova']['significant']}")
print(f"\nPeriod means:")
for period, mean_val in gdp_period_result["period_means"].items():
    print(f"  {period:15s}: {mean_val:>12.2f}")
print(f"\nPairwise t-tests:")
pw_df = pd.DataFrame(gdp_period_result["pairwise_tests"])
if not pw_df.empty:
    pw_df["sig_marker"] = pw_df["significant"].map({True: " ***", False: ""})
    display(pw_df[["period_a", "period_b", "t_statistic", "p_value", "mean_diff", "sig_marker"]]
            .rename(columns={"sig_marker": ""}))
print(f"\nInterpretation: {gdp_period_result['interpretation']}")

## 3. Unemployment Analysis

Unemployment is a lagging indicator that tells us how broadly economic stress is felt. We look at the overall trend, the 12-month moving average, the youth vs general gap, and crisis-period impacts.

In [ ]:
# ── Identify unemployment columns dynamically ───────────────────────────────
unemp_num = [c for c in unemployment.select_dtypes(include=[np.number]).columns if c not in _skip]
unemp_rate_col = next((c for c in unemp_num if "rate" in c.lower() and "youth" not in c.lower()), unemp_num[0])
unemp_youth_col = next((c for c in unemp_num if "youth" in c.lower()), None)

# Create a date proxy for plotting
unemployment["date_idx"] = unemployment["year"] + (unemployment["month"] - 1) / 12
unemployment["rolling_12m"] = unemployment[unemp_rate_col].rolling(window=12, min_periods=6).mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Unemployment trend with 12-month MA
ax = axes[0]
ax.plot(unemployment["date_idx"], unemployment[unemp_rate_col],
        alpha=0.35, color=CHART_PRIMARY, label="Monthly")
ax.plot(unemployment["date_idx"], unemployment["rolling_12m"],
        linewidth=2.5, color=CHART_ACCENT, label="12-month MA")
ax.axvspan(2012, 2014, alpha=0.10, color=CHART_ACCENT, label="Troika")
ax.axvspan(2020, 2021, alpha=0.10, color=ZONE_CAUTION, label="COVID")
ax.set_title("Unemployment Rate: Monthly + 12M Moving Average")
ax.set_xlabel("Year")
ax.set_ylabel("Rate (%)")
ax.legend(fontsize=CHART_FONT_SIZES["tick"])

# (b) Youth vs general unemployment scatter
ax = axes[1]
if unemp_youth_col:
    ax.scatter(unemployment[unemp_rate_col], unemployment[unemp_youth_col],
               alpha=0.5, s=18, color=CHART_SECONDARY, edgecolors="white", linewidths=0.3)
    # Regression line
    mask = unemployment[[unemp_rate_col, unemp_youth_col]].dropna()
    if len(mask) > 3:
        slope, intercept, r, p, _ = stats.linregress(mask[unemp_rate_col], mask[unemp_youth_col])
        xs = np.linspace(mask[unemp_rate_col].min(), mask[unemp_rate_col].max(), 50)
        ax.plot(xs, intercept + slope * xs, color=CHART_ACCENT, linewidth=1.5,
                label=f"OLS: y = {slope:.2f}x + {intercept:.1f} (R²={r**2:.2f})")
        ax.legend(fontsize=CHART_FONT_SIZES["tick"])
    ax.set_title("Youth vs General Unemployment")
    ax.set_xlabel("General Unemployment (%)")
    ax.set_ylabel("Youth Unemployment (%)")
else:
    ax.text(0.5, 0.5, "Youth unemployment\ncolumn not found", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Youth vs General Unemployment")

# (c) Crisis impact — bar chart of mean unemployment by period
ax = axes[2]
unemp_annual = unemployment.groupby("year")[unemp_rate_col].mean().reset_index()
period_means = {}
for pname, (s, e) in PERIODS.items():
    vals = unemp_annual.loc[(unemp_annual["year"] >= s) & (unemp_annual["year"] <= e), unemp_rate_col]
    if not vals.empty:
        period_means[pname] = vals.mean()
bars = ax.bar(period_means.keys(), period_means.values(),
              color=[period_color(p) for p in period_means.keys()],
              edgecolor="white")
ax.set_title("Mean Unemployment by Economic Period")
ax.set_ylabel("Rate (%)")
ax.tick_params(axis="x", rotation=25)
for bar, val in zip(bars, period_means.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
            f"{val:.1f}%", ha="center", fontsize=CHART_FONT_SIZES["legend"])

plt.tight_layout()
plt.savefig("../reports/ad_hoc/unemployment_analysis.png", bbox_inches="tight", dpi=CHART_DPI)
plt.show()
print(f"Primary unemployment column: {unemp_rate_col}")

## 4. Interest Rates and Credit

Interest rates drive the cost of borrowing. We look at the multi-rate environment (ECB, Euribor, sovereign yields), the sovereign spread, and the relationship between credit volume and non-performing loans.

In [ ]:
# ── Interest Rate columns ────────────────────────────────────────────────────
ir_num = [c for c in interest_rates.select_dtypes(include=[np.number]).columns if c not in _skip]
interest_rates["date_idx"] = interest_rates["year"] + (interest_rates["month"] - 1) / 12

ecb_col = next((c for c in ir_num if any(k in c.lower() for k in ["ecb", "refi", "key_rate", "main_refinancing"])), None)
sov_col = next((c for c in ir_num if any(k in c.lower() for k in ["sovereign", "bond", "yield", "10y"])), None)
eur_col = next((c for c in ir_num if "euribor" in c.lower()), None)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Multi-line rate plot
ax = axes[0]
rate_cols_found = [(ecb_col, "ECB Rate", CHART_ACCENT), (sov_col, "Sovereign Yield", CHART_PRIMARY),
                   (eur_col, "Euribor", CHART_SECONDARY)]
for col, label, color in rate_cols_found:
    if col:
        ax.plot(interest_rates["date_idx"], interest_rates[col], label=label, color=color, linewidth=1.5)
ax.axvspan(2012, 2014, alpha=0.10, color=CHART_ACCENT)
ax.axvspan(2020, 2021, alpha=0.10, color=ZONE_CAUTION)
ax.set_title("Interest Rate Environment (Portugal)")
ax.set_xlabel("Year")
ax.set_ylabel("Rate (%)")
ax.legend()
ax.axhline(0, color=CHART_NEUTRAL, linewidth=0.8, linestyle="--")

# (b) Sovereign spread (sovereign yield - ECB rate)
ax = axes[1]
if ecb_col and sov_col:
    interest_rates["sovereign_spread"] = interest_rates[sov_col] - interest_rates[ecb_col]
    ax.fill_between(interest_rates["date_idx"], interest_rates["sovereign_spread"],
                     alpha=0.4, color=ZONE_THRESHOLD, label="Sovereign spread")
    ax.plot(interest_rates["date_idx"], interest_rates["sovereign_spread"],
            color=CHART_ACCENT, linewidth=1.2)
    ax.axhline(0, color=CHART_NEUTRAL, linewidth=0.8, linestyle="--")
    ax.set_title("Portuguese Sovereign Spread over ECB Rate")
    ax.set_xlabel("Year")
    ax.set_ylabel("Spread (pp)")
    ax.legend()
else:
    ax.text(0.5, 0.5, "ECB or Sovereign column\nnot found", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig("../reports/interest_rates.png", bbox_inches="tight", dpi=CHART_DPI)
plt.show()

In [ ]:
# ── Credit and NPL relationship ──────────────────────────────────────────────
credit_num = [c for c in credit.select_dtypes(include=[np.number]).columns if c not in _skip]
credit_total_col = next((c for c in credit_num if any(k in c.lower() for k in ["total", "credit"])), credit_num[0] if credit_num else None)
npl_col = next((c for c in credit_num if any(k in c.lower() for k in ["npl", "non_performing", "nonperforming"])), None)

credit["date_idx"] = credit["year"] + (credit["month"] - 1) / 12

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Credit evolution
ax = axes[0]
if credit_total_col:
    ax.plot(credit["date_idx"], credit[credit_total_col], color=CHART_PRIMARY, linewidth=1.5, label="Total Credit")
    ax.set_title("Credit to the Economy")
    ax.set_xlabel("Year")
    ax.set_ylabel(credit_total_col.replace("_", " ").title())
    ax.axvspan(2012, 2014, alpha=0.10, color=CHART_ACCENT)
    ax.axvspan(2020, 2021, alpha=0.10, color=ZONE_CAUTION)
    ax.legend()

# (b) Credit vs NPL scatter
ax = axes[1]
if credit_total_col and npl_col:
    ax.scatter(credit[credit_total_col], credit[npl_col],
               alpha=0.5, s=20, c=credit["year"], cmap="RdYlBu_r", edgecolors="white", linewidths=0.3)
    ax.set_title("Credit Volume vs Non-Performing Loans")
    ax.set_xlabel(credit_total_col.replace("_", " ").title())
    ax.set_ylabel(npl_col.replace("_", " ").title())
    cbar = plt.colorbar(ax.collections[0], ax=ax)
    cbar.set_label("Year")
elif credit_total_col:
    credit_annual = credit.groupby("year")[credit_total_col].mean().reset_index()
    credit_annual["growth"] = credit_annual[credit_total_col].pct_change() * 100
    ax.bar(credit_annual["year"], credit_annual["growth"].fillna(0),
           color=CHART_SECONDARY, edgecolor="white")
    ax.axhline(0, color=CHART_DARK_TEXT, linewidth=0.8)
    ax.set_title("Annual Credit Growth Rate")
    ax.set_xlabel("Year")
    ax.set_ylabel("Growth (%)")

plt.tight_layout()
plt.savefig("../reports/credit_analysis.png", bbox_inches="tight", dpi=CHART_DPI)
plt.show()

## 5. Inflation and Public Debt

Inflation erodes purchasing power and affects debt sustainability. We examine headline inflation relative to the ECB 2% target, debt-to-GDP dynamics, and the fiscal balance evolution.

In [ ]:
# ── Inflation analysis ───────────────────────────────────────────────────────
inf_num = [c for c in inflation.select_dtypes(include=[np.number]).columns if c not in _skip]
inf_headline_col = next((c for c in inf_num if any(k in c.lower() for k in ["hicp", "cpi", "inflation", "headline"])), inf_num[0] if inf_num else None)
inf_core_col = next((c for c in inf_num if "core" in c.lower()), None)

inflation["date_idx"] = inflation["year"] + (inflation["month"] - 1) / 12

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Inflation vs ECB 2% target
ax = axes[0]
if inf_headline_col:
    ax.plot(inflation["date_idx"], inflation[inf_headline_col],
            color=CHART_PRIMARY, linewidth=1.2, alpha=0.7, label="Headline")
    if inf_core_col:
        ax.plot(inflation["date_idx"], inflation[inf_core_col],
                color=CHART_SECONDARY, linewidth=1.2, alpha=0.7, label="Core")
    ax.axhline(2.0, color=CHART_ACCENT, linestyle="--", linewidth=1.5, label="ECB 2% target")
    ax.axhline(0, color=CHART_NEUTRAL, linewidth=0.8, linestyle=":")
    ax.axvspan(2012, 2014, alpha=0.08, color=CHART_ACCENT)
    ax.axvspan(2020, 2021, alpha=0.08, color=ZONE_CAUTION)
    ax.set_title("Inflation vs ECB Target")
    ax.set_xlabel("Year")
    ax.set_ylabel("Annual rate (%)")
    ax.legend()

# (b) Inflation regime breakdown (pie chart)
ax = axes[1]
if inf_headline_col:
    vals = inflation[inf_headline_col].dropna()
    low = (vals < 1).sum()
    moderate = ((vals >= 1) & (vals <= 3)).sum()
    high = (vals > 3).sum()
    negative = (vals < 0).sum()
    sizes = [negative, low - negative, moderate, high]
    labels = [f"Deflation (<0%)\n{negative} months",
              f"Low (0-1%)\n{low - negative} months",
              f"Moderate (1-3%)\n{moderate} months",
              f"High (>3%)\n{high} months"]
    colors = [CHART_PRIMARY, CHART_SECONDARY, ZONE_CAUTION, CHART_ACCENT]
    sizes_clean = [s for s in sizes if s > 0]
    labels_clean = [l for l, s in zip(labels, sizes) if s > 0]
    colors_clean = [c for c, s in zip(colors, sizes) if s > 0]
    ax.pie(sizes_clean, labels=labels_clean, colors=colors_clean, autopct="%1.0f%%",
           startangle=140, textprops={"fontsize": CHART_FONT_SIZES["legend"]})
    ax.set_title("Inflation Regime Breakdown")

plt.tight_layout()
plt.savefig("../reports/inflation_analysis.png", bbox_inches="tight", dpi=CHART_DPI)
plt.show()

In [ ]:
# ── Public Debt: debt sustainability and fiscal balance ──────────────────────
debt_num = [c for c in public_debt.select_dtypes(include=[np.number]).columns if c not in _skip]
debt_gdp_col = next((c for c in debt_num if any(k in c.lower() for k in ["ratio", "gdp", "percent"])), None)
debt_abs_col = next((c for c in debt_num if any(k in c.lower() for k in ["debt", "total"]) and "ratio" not in c.lower() and "gdp" not in c.lower()), None)
fiscal_col = next((c for c in debt_num if any(k in c.lower() for k in ["balance", "deficit", "fiscal"])), None)

debt_primary = debt_gdp_col or debt_abs_col or (debt_num[0] if debt_num else None)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Debt-to-GDP ratio (or absolute debt) evolution
ax = axes[0]
if debt_primary:
    debt_annual = public_debt.groupby("year")[debt_primary].mean().reset_index()
    ax.plot(debt_annual["year"], debt_annual[debt_primary], marker="o", linewidth=2, color=CHART_ACCENT)
    ax.fill_between(debt_annual["year"], debt_annual[debt_primary], alpha=0.15, color=CHART_ACCENT)
    if debt_gdp_col:
        ax.axhline(60, color=CHART_POSITIVE, linestyle="--", linewidth=1.2, label="Maastricht 60% target")
        ax.axhline(100, color=ZONE_THRESHOLD, linestyle="--", linewidth=1.2, label="100% threshold")
    ax.axvspan(2012, 2014, alpha=0.08, color=CHART_ACCENT)
    ax.axvspan(2020, 2021, alpha=0.08, color=ZONE_CAUTION)
    ax.set_title(f"Public Debt Evolution ({debt_primary.replace('_', ' ').title()})")
    ax.set_xlabel("Year")
    ax.set_ylabel(debt_primary.replace("_", " ").title())
    ax.legend()

# (b) Fiscal balance evolution
ax = axes[1]
if fiscal_col:
    fiscal_annual = public_debt.groupby("year")[fiscal_col].mean().reset_index()
    colors_fiscal = [CHART_ACCENT if v < 0 else CHART_SECONDARY for v in fiscal_annual[fiscal_col]]
    ax.bar(fiscal_annual["year"], fiscal_annual[fiscal_col], color=colors_fiscal, edgecolor="white")
    ax.axhline(0, color=CHART_DARK_TEXT, linewidth=0.8)
    ax.axhline(-3, color=CHART_ACCENT, linestyle="--", linewidth=1, label="Maastricht -3% limit")
    ax.set_title("Fiscal Balance (% of GDP)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Balance (% GDP)")
    ax.legend()
elif debt_abs_col and debt_primary != debt_abs_col:
    debt_abs_annual = public_debt.groupby("year")[debt_abs_col].mean().reset_index()
    ax.bar(debt_abs_annual["year"], debt_abs_annual[debt_abs_col], color=CHART_SECONDARY, edgecolor="white")
    ax.set_title(f"Absolute Debt ({debt_abs_col.replace('_', ' ').title()})")
    ax.set_xlabel("Year")
    ax.set_ylabel("EUR millions")
else:
    ax.text(0.5, 0.5, "Fiscal balance column\nnot found", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig("../reports/public_debt_analysis.png", bbox_inches="tight", dpi=CHART_DPI)
plt.show()

## 6. Cross-Pillar Correlation Analysis

Economic indicators do not move in isolation. Here we build an annual cross-pillar dataset and explore key relationships — including the Phillips Curve (unemployment vs inflation) and debt-growth dynamics.

In [ ]:
# ── Build annual cross-pillar dataset ────────────────────────────────────────
# Aggregate each pillar to annual means for comparable time alignment.
cross = pd.DataFrame({"year": range(2010, 2026)})

# GDP
if gdp_val_col:
    cross = cross.merge(
        gdp.groupby("year")[gdp_val_col].mean().reset_index().rename(columns={gdp_val_col: "GDP"}),
        on="year", how="left")

# Unemployment
if unemp_rate_col:
    cross = cross.merge(
        unemployment.groupby("year")[unemp_rate_col].mean().reset_index().rename(columns={unemp_rate_col: "Unemployment"}),
        on="year", how="left")

# Inflation
if inf_headline_col:
    cross = cross.merge(
        inflation.groupby("year")[inf_headline_col].mean().reset_index().rename(columns={inf_headline_col: "Inflation"}),
        on="year", how="left")

# Interest rates (use first available)
if ir_num:
    ir_primary = ecb_col or ir_num[0]
    cross = cross.merge(
        interest_rates.groupby("year")[ir_primary].mean().reset_index().rename(columns={ir_primary: "Interest_Rate"}),
        on="year", how="left")

# Credit
if credit_total_col:
    cross = cross.merge(
        credit.groupby("year")[credit_total_col].mean().reset_index().rename(columns={credit_total_col: "Credit"}),
        on="year", how="left")

# Public Debt
if debt_primary:
    cross = cross.merge(
        public_debt.groupby("year")[debt_primary].mean().reset_index().rename(columns={debt_primary: "Debt"}),
        on="year", how="left")

cross = cross.set_index("year")
print(f"Cross-pillar dataset: {cross.shape[0]} years x {cross.shape[1]} indicators")
display(cross.round(2))

In [ ]:
# ── Correlation matrix heatmap ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Correlation heatmap
ax = axes[0]
corr = cross.corr()
mask_upper = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            mask=mask_upper, square=True, linewidths=0.8, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Cross-Pillar Correlation Matrix (Annual)")

# (b) Phillips Curve scatter (unemployment vs inflation)
ax = axes[1]
if "Unemployment" in cross.columns and "Inflation" in cross.columns:
    pc_data = cross[["Unemployment", "Inflation"]].dropna()
    ax.scatter(pc_data["Unemployment"], pc_data["Inflation"],
               s=60, c=pc_data.index, cmap="viridis", edgecolors="white", linewidths=0.5, zorder=5)
    # Add year labels
    for yr, row in pc_data.iterrows():
        ax.annotate(str(yr), (row["Unemployment"], row["Inflation"]),
                    fontsize=8, ha="left", va="bottom", xytext=(3, 3), textcoords="offset points")
    # Regression line
    if len(pc_data) >= 4:
        slope, intercept, r, p, _ = stats.linregress(pc_data["Unemployment"], pc_data["Inflation"])
        xs = np.linspace(pc_data["Unemployment"].min(), pc_data["Unemployment"].max(), 50)
        ax.plot(xs, intercept + slope * xs, color="red", linewidth=1.5, linestyle="--",
                label=f"OLS: r={r:.2f}, p={p:.3f}")
        ax.legend(fontsize=10)
    ax.set_title("Phillips Curve: Unemployment vs Inflation")
    ax.set_xlabel("Unemployment Rate (%)")
    ax.set_ylabel("Inflation Rate (%)")
    ax.axhline(2.0, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig("../reports/cross_pillar_correlation.png", bbox_inches="tight", dpi=150)
plt.show()

# Print key correlations
print("\nKey cross-pillar correlations:")
if corr.shape[0] >= 2:
    # Flatten and sort
    pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            r_val = corr.iloc[i, j]
            if not np.isnan(r_val):
                pairs.append((corr.columns[i], corr.columns[j], r_val))
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    for a, b, r_val in pairs:
        strength = "strong" if abs(r_val) > 0.7 else "moderate" if abs(r_val) > 0.4 else "weak"
        print(f"  {a:15s} vs {b:15s}:  r = {r_val:+.3f}  ({strength})")

## 7. Statistical Significance Tests

We go beyond descriptive statistics and test whether the patterns we see are statistically significant:
- **Trend tests** (linear regression) for each pillar
- **Period comparison** (ANOVA + Welch t-tests) across the five economic periods
- **Structural break tests** (Chow test) at the troika start (2012) and COVID shock (2020)
- **Granger proxy** (lagged correlation) for cross-pillar lead/lag relationships

In [ ]:
# ── Trend significance for each pillar ───────────────────────────────────────
pillar_map = {
    "GDP": (gdp, gdp_val_col, "year"),
    "Unemployment": (unemployment, unemp_rate_col, "year"),
    "Inflation": (inflation, inf_headline_col, "year"),
    "Credit": (credit, credit_total_col, "year"),
    "Public Debt": (public_debt, debt_primary, "year"),
}

# Add interest rate if available
if ir_num:
    pillar_map["Interest Rate"] = (interest_rates, ecb_col or ir_num[0], "year")

trend_rows = []
for name, (df, col, yr) in pillar_map.items():
    if col is None:
        continue
    annual = df.groupby(yr)[col].mean().dropna()
    result = test_trend_significance(annual.values, annual.index.values.astype(float))
    trend_rows.append({
        "Pillar": name,
        "Slope": result["slope"],
        "R-squared": result["r_squared"],
        "p-value": result["p_value"],
        "Significant": result["significant"],
        "Interpretation": result["interpretation"],
    })

trend_df = pd.DataFrame(trend_rows)
print("Trend Significance Tests (OLS linear regression)")
print("=" * 80)
display(trend_df.style.applymap(
    lambda v: "background-color: #d4edda" if v is True else ("background-color: #f8d7da" if v is False else ""),
    subset=["Significant"]
))

In [ ]:
# ── Period comparison (ANOVA) for each pillar ────────────────────────────────
anova_rows = []
for name, (df, col, yr) in pillar_map.items():
    if col is None:
        continue
    annual = df.groupby(yr)[col].mean().reset_index()
    result = test_period_comparison(annual, col, yr)
    anova_rows.append({
        "Pillar": name,
        "F-statistic": result["anova"]["f_statistic"],
        "p-value": result["anova"]["p_value"],
        "Significant": result["anova"]["significant"],
        "Sig. pairs": f"{result['n_significant_pairs']}/{result['n_total_pairs']}",
        "Interpretation": result["interpretation"],
    })

anova_df = pd.DataFrame(anova_rows)
print("Period Comparison (One-way ANOVA across 5 economic periods)")
print("=" * 80)
display(anova_df.style.applymap(
    lambda v: "background-color: #d4edda" if v is True else ("background-color: #f8d7da" if v is False else ""),
    subset=["Significant"]
))

In [ ]:
# ── Structural break tests (Chow test) ───────────────────────────────────────
break_rows = []
for name, (df, col, yr) in pillar_map.items():
    if col is None:
        continue
    annual = df.groupby(yr)[col].mean().dropna().reset_index()
    values = annual[col].values
    years = annual[yr].values
    for label, break_year in [("Troika (2012)", 2012), ("COVID (2020)", 2020)]:
        idx = int(np.searchsorted(years, break_year))
        result = test_structural_break(values, idx)
        break_rows.append({
            "Pillar": name,
            "Break Point": label,
            "F-statistic": result["f_statistic"],
            "p-value": result["p_value"],
            "Significant": result["significant"],
            "Interpretation": result["interpretation"],
        })

break_df = pd.DataFrame(break_rows)
print("Structural Break Tests (Chow Test Approximation)")
print("=" * 80)
display(break_df.style.applymap(
    lambda v: "background-color: #d4edda" if v is True else ("background-color: #f8d7da" if v is False else ""),
    subset=["Significant"]
))

In [ ]:
# ── Granger causality proxy (lagged correlations) ────────────────────────────
# Test whether unemployment leads GDP changes (and vice versa)
granger_pairs = []
if "GDP" in cross.columns and "Unemployment" in cross.columns:
    granger_pairs.append(("Unemployment", "GDP"))
    granger_pairs.append(("GDP", "Unemployment"))
if "Inflation" in cross.columns and "Interest_Rate" in cross.columns:
    granger_pairs.append(("Inflation", "Interest_Rate"))
    granger_pairs.append(("Interest_Rate", "Inflation"))
if "Unemployment" in cross.columns and "Inflation" in cross.columns:
    granger_pairs.append(("Unemployment", "Inflation"))
if "Debt" in cross.columns and "GDP" in cross.columns:
    granger_pairs.append(("Debt", "GDP"))

granger_results = []
for cause_name, effect_name in granger_pairs:
    cause_vals = cross[cause_name].dropna().values
    effect_vals = cross[effect_name].dropna().values
    min_len = min(len(cause_vals), len(effect_vals))
    result = test_granger_proxy(cause_vals[:min_len], effect_vals[:min_len], max_lag=5)
    granger_results.append({
        "Cause": cause_name,
        "Effect": effect_name,
        "Optimal Lag": result["optimal_lag"],
        "Correlation": result["correlation_at_lag"],
        "p-value": result["p_value"],
        "Interpretation": result["interpretation"],
    })

granger_df = pd.DataFrame(granger_results)
print("Granger Causality Proxy (Lagged Correlations)")
print("=" * 80)
display(granger_df)

In [ ]:
# ── Summary visualisation: significance dashboard ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Trend significance — bar chart of R-squared with significance markers
ax = axes[0]
colors_trend = [CHART_SECONDARY if row["Significant"] else CHART_NEUTRAL for _, row in trend_df.iterrows()]
bars = ax.barh(trend_df["Pillar"], trend_df["R-squared"], color=colors_trend, edgecolor="white")
ax.set_xlabel("R-squared")
ax.set_title("Trend Significance (R-squared)")
ax.axvline(0, color=CHART_NEUTRAL, linewidth=0.5)
for i, (_, row) in enumerate(trend_df.iterrows()):
    marker = " *" if row["Significant"] else ""
    ax.text(row["R-squared"] + 0.01, i, f'p={row["p-value"]:.3f}{marker}', va="center",
            fontsize=CHART_FONT_SIZES["tick"])

# (b) ANOVA F-statistics
ax = axes[1]
colors_anova = [CHART_SECONDARY if row["Significant"] else CHART_NEUTRAL for _, row in anova_df.iterrows()]
bars = ax.barh(anova_df["Pillar"], anova_df["F-statistic"].fillna(0), color=colors_anova, edgecolor="white")
ax.set_xlabel("F-statistic")
ax.set_title("Period Comparison (ANOVA F-statistic)")
for i, (_, row) in enumerate(anova_df.iterrows()):
    f_val = row["F-statistic"] if row["F-statistic"] else 0
    marker = " *" if row["Significant"] else ""
    ax.text(f_val + 0.1, i, f'p={row["p-value"]}{marker}', va="center",
            fontsize=CHART_FONT_SIZES["tick"])

# (c) Structural break p-values (heatmap-style)
ax = axes[2]
pivot = break_df.pivot(index="Pillar", columns="Break Point", values="p-value")
sig_pivot = break_df.pivot(index="Pillar", columns="Break Point", values="Significant")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn_r", vmin=0, vmax=0.2,
            linewidths=1, ax=ax, cbar_kws={"label": "p-value"})
ax.set_title("Structural Break p-values\n(green = not significant, red = significant)")

plt.tight_layout()
plt.savefig("../reports/significance_dashboard.png", bbox_inches="tight", dpi=CHART_DPI)
plt.show()

## 8. Conclusions

### Key Findings

1. **GDP has a significant upward trend** over the 2010-2025 period, recovering from the sovereign debt crisis through the troika programme and beyond, with a visible COVID dip in 2020.

2. **Unemployment peaked during the troika era** (2012-2014) and declined steadily during the recovery period. The youth unemployment rate consistently runs at roughly 2x the general rate, amplifying the social impact of economic downturns.

3. **Interest rates underwent a structural transformation** — from the sovereign debt crisis highs (when Portuguese bond yields spiked) through the ECB's zero/negative rate era, and into the 2022-2024 tightening cycle. The sovereign spread is a key barometer of market confidence.

4. **Inflation exhibited distinct regimes** — near-deflation during 2014-2016, stability around the ECB target during 2017-2019, and a sharp spike in 2022-2023 driven by energy and supply-chain pressures.

5. **Public debt remains elevated** relative to the Maastricht 60% target, though the trajectory since 2014 has been one of gradual consolidation (interrupted by COVID).

6. **Cross-pillar relationships confirm textbook macroeconomic patterns** — the Phillips Curve (unemployment-inflation trade-off), interest rate transmission to credit, and the debt-growth feedback loop are all visible in the data.

### Caveats

- The data in this project is **synthetic / simulated** for portfolio demonstration purposes. The patterns are designed to be realistic but do not represent actual official statistics.
- With only 16 annual observations (2010-2025), statistical power is limited. The monthly/quarterly granularity helps, but cross-pillar comparisons at the annual level should be interpreted cautiously.
- Correlation does not imply causation. The Granger proxy tests are suggestive but not definitive.

### Next Steps

- Expand the analysis with **forecasting models** (ARIMA, VAR) to project key indicators.
- Build an **interactive Power BI dashboard** for real-time exploration.
- Generate **AI-powered narrative insights** using the GPT-4 integration module.
- Add **scenario analysis** (e.g., what if ECB raises rates by 100bp?).

---
*Source: Portugal Data Intelligence project | Data: Synthetic (based on realistic Portuguese macroeconomic patterns)*

In [ ]:
# ── Cleanup ──────────────────────────────────────────────────────────────────
conn.close()
print("Database connection closed. Analysis complete.")